# de Bruijn-Sharma Problem

## What AlphaEvolve found

AlphaEvolve was tasked with producing polynomials that excluded various half-planes of pairs $(\alpha, \beta)$ as infeasible for the de Bruijn-Sharma inequality. To the authors' surprise, AlphaEvolve indicated that the feasible region $\Omega(n)$ was slightly larger than initially believed: the $x$-intercept could be improved to $\left(\frac{n^3 - 2n^2 + 3n - 14}{n(n^2 + 3)}, 0\right)$ when $n$ was odd, and the $y$-intercept to $\left(0, \frac{(n-2)^4 + n-2}{n^2(n-1)^2}\right)$ for all $n \geq 4$. These improvements came from the polynomials used by AlphaEvolve exploiting the constraint that the zeroes sum to zero. The paper notes: "this experiment provided an example in which AlphaEvolve was able to notice an oversight in the analysis by the human authors." These conjectured inequalities were subsequently proved by Tang.

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order


def _get_halfspace_from_poly(
    roots: np.ndarray, n: int
) -> Tuple[float, float, float] | None:
  """Calculates the C4, C2_sq, and LHS terms for a given polynomial."""
  if roots.shape[0] != n or np.any(np.isnan(roots)):
    return None
  if np.abs(np.sum(roots)) > 1e-9:
    return None
  try:
    p_coeffs = np.poly(roots)
    if np.any(np.isnan(p_coeffs)):
      return None
    p_prime_coeffs = np.polyder(p_coeffs)
    if p_prime_coeffs.size == 0:
      return None
    critical_points = np.roots(p_prime_coeffs)

    # --- CRUCIAL CHANGE: Use sum of FOURTH powers as per the paper ---
    lhs = np.sum(np.abs(critical_points) ** 4)

    c4 = np.sum(np.abs(roots) ** 4)
    c2_sq = (np.sum(np.abs(roots) ** 2)) ** 2
    if np.isnan(lhs) or np.isnan(c4) or np.isnan(c2_sq):
      return None
    if c4 < 1e-9 and c2_sq < 1e-9:
      return None
    return (c4, c2_sq, lhs)
  except np.linalg.LinAlgError:
    return None


def calculate_area_score(polynomials: List[np.ndarray], n: int) -> float:
  """Calculates the score, which is the negative area of the feasible region."""
  if not isinstance(polynomials, list) or not polynomials:
    return -4.0

  interior_point = np.array([0.75, 0.75])
  halfspaces = [
      [-1.0, 0.0, 0.0],
      [-0.0, -1.0, 0.0],
      [1.0, 0.0, -1.0],
      [0.0, 1.0, -1.0],
  ]

  for poly_roots in polynomials:
    params = _get_halfspace_from_poly(poly_roots, n)
    if params:
      c4, c2_sq, lhs = params
      halfspaces.append([-c4, -c2_sq, lhs])

  # if len(halfspaces) <= 4: return -4.0

  try:
    # With the correct formula, the interior point should be stable
    vertices = HalfspaceIntersection(
        np.array(halfspaces), interior_point
    ).intersections
    if vertices.shape[0] < 3:
      return 0.0
    return -ConvexHull(vertices).volume - len(polynomials) / 10000.0
  except Exception as e:
    print(f'Warning: Qhull error encountered, returning poor score. {e}')
    return -1_000_000.0


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation in code."""
  formatted_feedback = {}
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    elif isinstance(value, list):
      formatted_feedback[key] = repr(value)  # Use standard repr for lists
    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(params: int) -> Tuple[Dict[str, float], Dict[str, str]]:
  """Evaluates a set of polynomials and plots the result."""
  result = {}
  feedback = {}
  n = params

  best_polynomials_found = search_for_best_polynomials(n)
  score = calculate_area_score(best_polynomials_found, n)

  result['score'] = score
  feedback['best_score_found'] = score
  feedback['best_polynomials'] = str(best_polynomials_found)
  feedback = format_feedback_repr(feedback)

  return result, feedback

In [ ]:
#@title Initial program

"""FunSearch experiment for the de Bruin-Sharma conjecture with REAL ROOTS,

using the correct sum of fourth powers.
"""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import random
import re
from typing import Any, Callable, Mapping, List, Tuple, Dict
import scipy.linalg as la
import collections
import copy
import math
import numba
from scipy.spatial import HalfspaceIntersection, ConvexHull

njit = numba.njit



def search_for_best_polynomials(n: int) -> List[np.ndarray]:
  """Searches for a set of polynomials with complex roots that minimizes the

  feasible region.
  """
  variable_name = f'best_polynomials_{n}'

  if variable_name in globals() and globals()[variable_name]:
    best_polynomials = [p.copy() for p in globals()[variable_name]]
  else:
    # Generate complex roots in conjugate pairs to ensure real coefficients.
    num_pairs = n // 2
    complex_part = np.random.randn(num_pairs, 2).view(np.complex128).flatten()
    roots = np.concatenate([complex_part, np.conj(complex_part)])
    if n % 2 != 0:  # If n is odd, add a real root.
      roots = np.append(roots, 0.0)
    # Enforce the zero-sum constraint. This preserves the conjugate pairs.
    roots -= np.mean(roots)
    best_polynomials = [roots]

  best_score = calculate_area_score(best_polynomials, n)

  current_polynomials = [p.copy() for p in best_polynomials]

  start_time = time.time()
  eval_count = 0
  while time.time() - start_time < 100:
    mutation_type = np.random.rand()
    temp_polys = [p.copy() for p in current_polynomials]

    if not temp_polys:  # Failsafe
      num_pairs = n // 2
      complex_part = np.random.randn(num_pairs, 2).view(np.complex128).flatten()
      new_roots = np.concatenate([complex_part, np.conj(complex_part)])
      if n % 2 != 0:
        new_roots = np.append(new_roots, 0.0)
      new_roots -= np.mean(new_roots)
      temp_polys.append(new_roots)

    if mutation_type < 0.9:  # Mutate a polynomial
      poly_idx = np.random.randint(len(temp_polys))
      poly = temp_polys[poly_idx]

      # Pick two distinct roots to perturb.
      idx1, idx2 = np.random.choice(n, 2, replace=False)
      z1_old, z2_old = poly[idx1], poly[idx2]

      # Find the indices of their conjugates.
      conj_idx1 = np.argmin(np.abs(poly - np.conj(z1_old)))
      conj_idx2 = np.argmin(np.abs(poly - np.conj(z2_old)))

      # Generate a small complex mutation.
      mutation = (np.random.randn() + 1j * np.random.randn()) * 0.002

      # Apply a 4-way mutation to preserve both the zero-sum and
      # conjugate-pair properties. This ensures the resulting
      # polynomial still has real coefficients.
      poly[idx1] += mutation
      poly[idx2] -= mutation
      poly[conj_idx1] += np.conj(mutation)
      poly[conj_idx2] -= np.conj(mutation)

    elif mutation_type < 0.95:  # Add a new polynomial
      num_pairs = n // 2
      complex_part = np.random.randn(num_pairs, 2).view(np.complex128).flatten()
      new_roots = np.concatenate([complex_part, np.conj(complex_part)])
      if n % 2 != 0:
        new_roots = np.append(new_roots, 0.0)
      new_roots -= np.mean(new_roots)  # Enforce zero-sum.
      temp_polys.append(new_roots)
      if len(temp_polys) > 25:
        temp_polys.pop(np.random.randint(len(temp_polys)))
    else:  # Remove a polynomial
      if len(temp_polys) > 1:
        temp_polys.pop(np.random.randint(len(temp_polys)))

    current_polynomials = temp_polys
    score = calculate_area_score(current_polynomials, n)
    eval_count += 1

    if score > best_score:
      best_score = score
      best_polynomials = [p.copy() for p in current_polynomials]
      print(
          f'New best score: {best_score} with'
          f' {len(best_polynomials)} polynomials.'
      )

    if np.random.rand() < 0.2:
      current_polynomials = [p.copy() for p in best_polynomials]

  return best_polynomials

Act as an expert in complex analysis and computational geometry. Your goal is to help solve the de Bruin-Sharma problem by finding polynomials that provide the tightest possible constraints.

The problem seeks the set of pairs $(\alpha, \beta) \in \mathbb(R)+^2$ such that for any degree $n$ polynomial $P$ whose roots $z_1, \dots, z_n$ sum to zero, the following inequality holds:
$$ |\xi_1|^2 + \dots + |\xi(n-1)|^2 \leq \alpha (|z_1|^4 + \dots + |z_n|^4) + \beta (|z_1|^2 + \dots + |z_n|^2)^2 $$
where $\xi_1, \dots, \xi_(n-1)$ are the critical points of $P$ (the roots of its derivative $P'$).

Each polynomial you find defines a linear constraint on $(\alpha, \beta)$, effectively cutting off a half-plane from the space of possible pairs. The true valid region, $\Omega(n)$, is the intersection of all such half-planes generated by all possible polynomials.

Your task is to write a search function that discovers a list of polynomials that collectively define the smallest possible valid region for $(\alpha, \beta)$. Your function will be given the polynomial degree, n. Each polynomial must be represented by its roots, which must be a list or NumPy array of $n$ complex numbers that sum to zero.

Your list of polynomials will be evaluated by the following scoring function, which calculates the area of the feasible $(\alpha, \beta)$ region within the unit square $[0,1] \times [0,1]$ defined by your polynomials. Since the goal is to minimize this area, and FunSearch always maximizes, the score is the negative of the area. A higher score (closer to zero) is better.

Here is the evaluation code you will be scored against:

def _get_halfspace_from_poly(
    roots: np.ndarray, n: int
) -> Tuple[float, float, float] | None:
    """Calculates the parameters for the half-space inequality from polynomial roots."""
    if roots.shape[0] != n or np.any(np.isnan(roots)):
        return None
    if np.abs(np.sum(roots)) > 1e-9:
        return None
    try:
        p_coeffs = np.poly(roots)
        if np.any(np.isnan(p_coeffs)): return None
        p_prime_coeffs = np.polyder(p_coeffs)
        if p_prime_coeffs.size == 0: return None
        critical_points = np.roots(p_prime_coeffs)
        lhs = np.sum(np.abs(critical_points) ** 2)
        c4 = np.sum(np.abs(roots) ** 4)
        c2_sq = (np.sum(np.abs(roots) ** 2)) ** 2
        if np.isnan(lhs) or np.isnan(c4) or np.isnan(c2_sq): return None
        if c4 < 1e-9 and c2_sq < 1e-9: return None
        return (c4, c2_sq, lhs)
    except np.linalg.LinAlgError:
        return None

def calculate_area_score(polynomials: List[np.ndarray], n: int) -> float:
    """Calculates the score, which is the negative area of the feasible (alpha, beta) region."""
    if not isinstance(polynomials, list) or not polynomials:
        return -1_000_000.0

A = [[-1.0, 0.0], [0.0, -1.0], [1.0, 0.0], [0.0, 1.0]]
b = [0.0, 0.0, 1.0, 1.0]

for poly_roots in polynomials:
    params = _get_halfspace_from_poly(poly_roots, n)
    if params:
        c4, c2_sq, lhs = params
        A.append([-c4, -c2_sq])
        b.append(-lhs)
try:
    interior_point = np.array([0.5, 0.5])
    halfspace_intersection = HalfspaceIntersection(
        np.array(A), np.array(b), interior_point=interior_point
    )
    vertices = halfspace_intersection.intersections
    if vertices.shape[0] < 3: return 0.0
    hull = ConvexHull(vertices)
    area = hull.volume
    if np.isnan(area) or area < 0: return -1_000_000.0
    return -area
except Exception:
    return -1_000_000.0
You can call this scoring function as often as you like to guide your search. Your search function has 100 seconds to find the best possible list of polynomials before it must return its result. Good luck!

